# Feature Engineering for Clinical Events, Labs, and Encounters

## Objectives
After completing this lab, you will be able to:
- Engineer features from raw clinical, lab, and encounter data.  
- Aggregate lab values across clinically meaningful time windows.  
- Derive encounter frequency and utilization indicators.  
- Encode comorbidities and medication counts.  
- Construct episode-level features while avoiding data leakage.


## What you will do in this lab
In this lab, you will prepare longitudinal clinical data for modeling by aggregating events, encoding clinical context, and preventing data leakage.

You will:
- Load a synthetic longitudinal clinical dataset.  
- Aggregate laboratory results over time windows.  
- Derive encounter-based utilization features.  
- Encode comorbidities and medication exposure.  
- Build episode-level features suitable for modeling.  
- Identify and avoid common data leakage pitfalls.


## Overview
Raw clinical data must be transformed into meaningful features before it can be used
for analytics or machine learning. Feature engineering in healthcare requires careful
attention to time, clinical context, and patient-level aggregation. This lab focuses
on creating clinically sensible features from events, labs, and encounters while
highlighting common pitfalls such as data leakage.


## About the dataset/environment
You will work with a **synthetic longitudinal clinical dataset** that includes:
- Patient encounters over time  
- Laboratory test results  
- Medication records  
- Diagnosis codes  

The dataset is designed to support feature engineering exercises and includes
timestamps needed for time-aware aggregation.


## Setup

In [1]:

# This cell imports required libraries and loads a synthetic longitudinal clinical dataset.


import pandas as pd


# Load a synthetic encounter-level dataset
encounters_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab5/encounters_dataset1.csv")

# Load a synthetic lab dataset
labs_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab5/laboratory_dataset1.csv")

# Load a synthetic medication dataset
meds_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab5/medications_dataset1.csv")


In [2]:
encounters_df.head()

,patient_id,encounter_id,encounter_date,diagnosis_code
0,P0001,EP0001_1,2023-02-21,I10
1,P0001,EP0001_2,2023-03-13,E11
2,P0001,EP0001_3,2023-04-06,I10
3,P0001,EP0001_4,2023-04-10,E11
4,P0001,EP0001_5,2023-04-10,I10


In [3]:
labs_df.head()

,patient_id,lab_date,lab_name,lab_value
0,P0001,2023-01-14,Glucose,157.0
1,P0001,2023-01-28,Glucose,102.3
2,P0001,2023-02-11,Glucose,90.5
3,P0001,2023-03-12,Glucose,140.9
4,P0002,2023-01-02,Glucose,137.2


In [4]:
meds_df.head()

,patient_id,medication_name,med_start_date
0,P0001,Lisinopril,2023-01-09
1,P0001,Metformin,2023-02-21
2,P0001,Lisinopril,2023-03-12
3,P0002,Lisinopril,2022-12-27
4,P0002,Lisinopril,2023-01-15


## Step 1: Aggregate lab values over time windows
You will begin by summarizing lab values over a fixed time window before each encounter.
This allows capturing recent clinical status.

**Why this matters in healthcare:** Lab trends over time are often more informative than single measurements.


In [5]:
# This cell aggregates lab values within 30 days prior to each encounter.
# Time-windowed aggregation is a common clinical feature engineering technique.

# Convert date columns to datetime objects
encounters_df['encounter_date'] = pd.to_datetime(encounters_df['encounter_date'])
labs_df['lab_date'] = pd.to_datetime(labs_df['lab_date'])

# Merge encounters with labs
enc_labs = encounters_df.merge(labs_df, on="patient_id", how="left")

# Calculate days between lab and encounter
enc_labs["days_before_encounter"] = (
    enc_labs["encounter_date"] - enc_labs["lab_date"]
).dt.days

# Filter labs within 30 days before encounter
recent_labs = enc_labs[
    (enc_labs["days_before_encounter"] >= 0) &
    (enc_labs["days_before_encounter"] <= 30)
]

# Aggregate mean lab value per encounter
lab_features = recent_labs.groupby("encounter_id")["lab_value"].mean().reset_index()
lab_features

,encounter_id,lab_value
0,EP0001_1,96.4
1,EP0001_2,115.7
2,EP0001_3,140.9
3,EP0001_4,140.9
4,EP0001_5,140.9
...,...,...
1084,EP0397_2,118.8
1085,EP0397_3,122.0
1086,EP0397_4,122.0
1087,EP0399_1,117.1


## Step 2: Derive encounter frequency features
Here, you will calculate how frequently patients are visiting healthcare facilities.

**Why this matters in healthcare:** High encounter frequency often signals disease severity or care complexity.


In [6]:

# This cell derives encounter frequency per patient.
# Utilization features are strong predictors in many healthcare models.

encounter_counts = encounters_df.groupby("patient_id")["encounter_id"]                                  .count()                                  .reset_index(name="encounter_count")

encounter_counts


,patient_id,encounter_count
0,P0001,5
1,P0002,5
2,P0003,6
3,P0004,5
4,P0005,6
...,...,...
395,P0396,6
396,P0397,4
397,P0398,3
398,P0399,3


## Step 3: Encode comorbidities
You will convert diagnosis codes into indicators representing the presence of chronic conditions.

**Why this matters in healthcare:** Comorbidities strongly influence outcomes and risk stratification.


In [7]:

# This cell encodes comorbidities as binary indicators.
# Diagnosis-based features are commonly used in clinical models.

# Create comorbidity flags
encounters_df["has_diabetes"] = encounters_df["diagnosis_code"] == "E11"
encounters_df["has_hypertension"] = encounters_df["diagnosis_code"] == "I10"

# Aggregate to patient level
comorbidity_features = encounters_df.groupby("patient_id")[
    ["has_diabetes", "has_hypertension"]
].any().reset_index()

comorbidity_features


,patient_id,has_diabetes,has_hypertension
0,P0001,True,True
1,P0002,True,True
2,P0003,True,True
3,P0004,True,True
4,P0005,True,True
...,...,...,...
395,P0396,True,True
396,P0397,True,True
397,P0398,True,True
398,P0399,False,True


## Step 4: Derive medication exposure features
Next, you will count the number of medications prescribed to each patient.

**Why this matters in healthcare:** Medication burden is a proxy for disease complexity and risk.


In [8]:

# This cell counts distinct medications per patient.
# Medication exposure features are important clinical predictors.

med_counts = meds_df.groupby("patient_id")["medication_name"].nunique().reset_index(name="medication_count")

med_counts


,patient_id,medication_count
0,P0001,2
1,P0002,1
2,P0003,1
3,P0004,1
4,P0005,2
...,...,...
395,P0396,2
396,P0397,2
397,P0398,1
398,P0399,2


## Step 5: Construct episode-level features
You will now combine all engineered features into a single episode-level dataset.

**Why this matters in healthcare:** Models typically operate on episode- or patient-level feature tables.


In [9]:

# This cell merges all engineered features.
# Consolidating features prepares the dataset for modeling.

episode_features = encounters_df.merge(lab_features, on="encounter_id", how="left").merge(encounter_counts, on="patient_id", how="left").merge(comorbidity_features, on="patient_id", how="left").merge(med_counts, on="patient_id", how="left")

episode_features


,patient_id,encounter_id,encounter_date,diagnosis_code,has_diabetes_x,has_hypertension_x,lab_value,encounter_count,has_diabetes_y,has_hypertension_y,medication_count
0,P0001,EP0001_1,2023-02-21,I10,False,True,96.4,5,True,True,2
1,P0001,EP0001_2,2023-03-13,E11,True,False,115.7,5,True,True,2
2,P0001,EP0001_3,2023-04-06,I10,False,True,140.9,5,True,True,2
3,P0001,EP0001_4,2023-04-10,E11,True,False,140.9,5,True,True,2
4,P0001,EP0001_5,2023-04-10,I10,False,True,140.9,5,True,True,2
...,...,...,...,...,...,...,...,...,...,...,...
1846,P0400,EP0400_1,2023-02-24,I10,False,True,92.6,5,True,True,1
1847,P0400,EP0400_2,2023-03-18,E11,True,False,NaN,5,True,True,1
1848,P0400,EP0400_3,2023-04-05,E11,True,False,NaN,5,True,True,1
1849,P0400,EP0400_4,2023-04-01,I10,False,True,NaN,5,True,True,1


## Step 6: Identify and avoid data leakage
Finally, you will review features to ensure they do not use information from the future.

**Why this matters in healthcare:** Data leakage produces misleadingly high model performance and unsafe models.


In [10]:

# This cell highlights potential data leakage risks.
# Features must only use information available at prediction time.

# Example leakage check: ensure lab dates precede encounter dates
leakage_check = enc_labs[enc_labs["days_before_encounter"] < 0]

leakage_check


,patient_id,encounter_id,encounter_date,diagnosis_code,lab_date,lab_name,lab_value,days_before_encounter
3,P0001,EP0001_1,2023-02-21,I10,2023-03-12,Glucose,140.9,-19
21,P0002,EP0002_1,2023-01-11,I10,2023-01-15,Glucose,130.5,-4
22,P0002,EP0002_1,2023-01-11,I10,2023-02-09,Glucose,110.6,-29
23,P0002,EP0002_1,2023-01-11,I10,2023-02-01,Glucose,134.4,-21
24,P0002,EP0002_1,2023-01-11,I10,2023-01-30,Glucose,66.2,-19
...,...,...,...,...,...,...,...,...
8180,P0396,EP0396_5,2023-03-15,I10,2023-04-23,Glucose,122.0,-39
8187,P0397,EP0397_1,2023-01-08,E11,2023-01-09,Glucose,120.8,-1
8188,P0397,EP0397_1,2023-01-08,E11,2023-01-27,Glucose,116.8,-19
8189,P0397,EP0397_1,2023-01-08,E11,2023-02-12,Glucose,127.2,-35


## Exercises

In [11]:
# Load a synthetic encounter-level dataset for exercises
encounters_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab5/encounters_dataset2.csv")

# Load a synthetic lab dataset for exercises
labs_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab5/laboratory_dataset2.csv")

# Load a synthetic medication dataset for exercises
meds_df = pd.read_csv("https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab5/medications_dataset2.csv")

### Exercise 1: Aggregate lab values over time windows

In [12]:
# your code goes here

# Convert date columns to datetime objects
encounters_df['encounter_date'] = pd.to_datetime(encounters_df['encounter_date'])
labs_df['lab_date'] = pd.to_datetime(labs_df['lab_date'])

# Merge encounters with labs
enc_labs = encounters_df.merge(labs_df, on="patient_id", how="left")

# Calculate days between lab and encounter
enc_labs["days_before_encounter"] = (
    enc_labs["encounter_date"] - enc_labs["lab_date"]
).dt.days

# Filter labs within 30 days before encounter
recent_labs = enc_labs[
    (enc_labs["days_before_encounter"] >= 0) &
    (enc_labs["days_before_encounter"] <= 30)
]

# Aggregate mean lab value per encounter
lab_features = recent_labs.groupby("encounter_id")["lab_value"].mean().reset_index()
lab_features

,encounter_id,lab_value
0,E00001,87.880
1,E00005,96.540
2,E00006,87.905
3,E00007,8.360
4,E00009,65.040
...,...,...
737,E01883,67.175
738,E01886,161.590
739,E01889,99.850
740,E01890,134.640


### Exercise 2: Calculate encounter frequency

In [13]:
# your code goes here
encounter_counts = encounters_df.groupby("patient_id")["encounter_id"].count().reset_index(name="encounter_count")
encounter_counts

,patient_id,encounter_count
0,PT0001,7
1,PT0002,6
2,PT0003,9
3,PT0004,9
4,PT0005,6
...,...,...
295,PT0296,5
296,PT0297,9
297,PT0298,4
298,PT0299,10


### Exercise 3: Encode comorbidities

In [14]:
# your code goes here

# Create comorbidity flags
encounters_df["has_diabetes"] = encounters_df["diagnosis_code"] == "E11"
encounters_df["has_hypertension"] = encounters_df["diagnosis_code"] == "I10"

# Aggregate to patient level
comorbidity_features = encounters_df.groupby("patient_id")[
    ["has_diabetes", "has_hypertension"]
].any().reset_index()

comorbidity_features

,patient_id,has_diabetes,has_hypertension
0,PT0001,True,False
1,PT0002,True,True
2,PT0003,True,True
3,PT0004,True,True
4,PT0005,True,True
...,...,...,...
295,PT0296,False,True
296,PT0297,True,True
297,PT0298,True,False
298,PT0299,True,True


### Exercise 4: Derive medication counts

In [15]:
# your code goes here

med_counts = meds_df.groupby("patient_id")["medication_name"].nunique().reset_index(name="medication_count")

med_counts

,patient_id,medication_count
0,PT0001,3
1,PT0002,3
2,PT0003,2
3,PT0004,2
4,PT0005,1
...,...,...
279,PT0296,2
280,PT0297,2
281,PT0298,2
282,PT0299,2


### Exercise 5: Build episode-level features

In [17]:
# your code goes here

episode_features = encounters_df.merge(lab_features, on="encounter_id", how="left").merge(encounter_counts, on="patient_id", how="left").merge(comorbidity_features, on="patient_id", how="left").merge(med_counts, on="patient_id", how="left")

episode_features

,patient_id,encounter_id,encounter_date,diagnosis_code,has_diabetes_x,has_hypertension_x,lab_value,encounter_count,has_diabetes_y,has_hypertension_y,medication_count
0,PT0099,E00001,2023-06-20,E78,False,False,87.88,4,False,True,2.0
1,PT0231,E00002,2023-10-05,E11,True,False,NaN,8,True,True,2.0
2,PT0018,E00003,2023-08-14,E78,False,False,NaN,11,True,True,3.0
3,PT0084,E00004,2023-06-10,I10,False,True,NaN,6,True,True,2.0
4,PT0107,E00005,2023-04-18,E11,True,False,96.54,8,True,True,3.0
...,...,...,...,...,...,...,...,...,...,...,...
1895,PT0155,E01896,2023-07-04,E11,True,False,NaN,9,True,True,2.0
1896,PT0087,E01897,2023-04-16,I10,False,True,NaN,8,True,True,3.0
1897,PT0226,E01898,2023-02-22,J45,False,False,NaN,9,True,True,2.0
1898,PT0123,E01899,2023-02-14,E78,False,False,NaN,6,True,False,2.0


### Exercise 6: Check for data leakage

In [18]:
# your code goes here

leakage_check = enc_labs[enc_labs["days_before_encounter"] < 0]

leakage_check

,patient_id,encounter_id,encounter_date,diagnosis_code,lab_date,lab_name,lab_value,days_before_encounter
0,PT0099,E00001,2023-06-20,E78,2023-08-30,LDL,106.60,-71
3,PT0099,E00001,2023-06-20,E78,2023-10-16,Glucose,154.47,-118
4,PT0099,E00001,2023-06-20,E78,2023-10-09,LDL,28.95,-111
10,PT0231,E00002,2023-10-05,E11,2023-11-02,HbA1c,65.24,-28
11,PT0018,E00003,2023-08-14,E78,2023-10-02,Creatinine,115.67,-49
...,...,...,...,...,...,...,...,...
11475,PT0123,E01899,2023-02-14,E78,2023-12-25,HbA1c,43.75,-314
11476,PT0123,E01899,2023-02-14,E78,2023-04-30,Glucose,165.95,-75
11477,PT0123,E01899,2023-02-14,E78,2023-11-27,Glucose,55.80,-286
11478,PT0123,E01899,2023-02-14,E78,2023-06-19,Glucose,111.23,-125
